In [2]:
import pickle
import torch
import torch.nn as nn
import pandas as pd
from pykeen.triples import TriplesFactory
from pykeen.models import ERModel
from pykeen.nn import DistMultInteraction, TransEInteraction, RotatEInteraction
from pykeen.models import DistMult
from pykeen.training import SLCWATrainingLoop
from pykeen.losses import MarginRankingLoss
from pykeen.evaluation import RankBasedEvaluator, SampledRankBasedEvaluator
from torch.optim import Adam, RMSprop, NAdam


In [3]:
main_data = pd.read_csv('data/edges/IW_edges.csv')
main_data = main_data.astype(str)

triples = main_data[['id_entity_1', 'relation', 'id_entity_2']].values
triplet_data = TriplesFactory.from_labeled_triples(triples, create_inverse_triples=True)
training_set, testing_set, validation_set = triplet_data.split([0.8, 0.1, 0.1], random_state=17)



In [5]:
EMB_DIM = 128
MARGIN = 1.1
LR = 1e-3
NUM_NEGS_PER_POS = 15
EPOCHS = 5
BATCH_SIZE = 4096
WEIGHT = 1e-3
device = 'cuda'

loss_function = MarginRankingLoss(margin=MARGIN)
model = DistMult(
    triples_factory=training_set,
    embedding_dim=EMB_DIM,
    random_seed=100,
    loss = loss_function,
)
model = model.to(device)

optimizer = Adam(params=model.get_grad_params(), lr=LR)

training_loop = SLCWATrainingLoop(
    model=model,
    triples_factory=training_set,
    optimizer=optimizer,
    negative_sampler='pseudotyped',
    negative_sampler_kwargs=dict(
        num_negs_per_pos=NUM_NEGS_PER_POS
    )
)

evaluator = RankBasedEvaluator()

training_loop.train(
    num_epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    triples_factory=training_set,
    use_tqdm_batch=False,
)

model_results = evaluator.evaluate(
    model=model,
    mapped_triples=testing_set.mapped_triples[:1500].to(device),
    additional_filter_triples=[
            training_set.mapped_triples.to(device),
            validation_set.mapped_triples.to(device),
        ],
)

metrics = model_results.to_df()
metrics = metrics[(metrics['Side'] == 'both') & (metrics['Rank_type'] == 'realistic')]
metrics


Training epochs on cuda:0: 100%|██████████| 5/5 [02:43<00:00, 32.73s/epoch, loss=1.1, prev_loss=1.1]
Evaluating on cuda:0: 100%|██████████| 1.50k/1.50k [00:07<00:00, 191triple/s]


,Side,Rank_type,Metric,Value
5,both,realistic,inverse_arithmetic_mean_rank,3.733912e-06
14,both,realistic,adjusted_arithmetic_mean_rank_index,-2.711786e-03
23,both,realistic,inverse_median_rank,3.817624e-06
32,both,realistic,z_arithmetic_mean_rank,-2.572619e-01
41,both,realistic,median_rank,2.619430e+05
50,both,realistic,arithmetic_mean_rank,2.678157e+05
59,both,realistic,adjusted_geometric_mean_rank_index,-1.120885e-02
68,both,realistic,adjusted_arithmetic_mean_rank,1.002712e+00
77,both,realistic,z_geometric_mean_rank,-6.141404e-01
86,both,realistic,count,3.000000e+03
